# Benchmarks

## Initialize

In [1]:
import os
import math
import pathlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import clear_output
import ray

import warnings
import lifelines
from lifelines.utils import CensoringType
from lifelines.utils import concordance_index

In [2]:
lifelines.__version__

'0.26.4'

In [3]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

experiment = 220413
experiment_path = f"{output_path}/{experiment}"
pathlib.Path(experiment_path).mkdir(parents=True, exist_ok=True)

/sc-projects/sc-proj-ukb-cvd


In [4]:
data_outcomes = pd.read_feather(f"{output_path}/baseline_outcomes_220412.feather").set_index("eid")

In [5]:
endpoints_md = pd.read_csv(f"{experiment_path}/endpoints.csv")
endpoints = sorted(endpoints_md.endpoint.to_list())

In [6]:
partitions = [i for i in range(22)]#[0, 1, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
splits = ["train", "valid", "test"]

In [7]:
endpoint_defs = pd.read_feather(f"{output_path}/phecode_defs_220306.feather").query("endpoint==@endpoints").sort_values("endpoint").set_index("endpoint")

In [8]:
eligable_eids = pd.read_feather(f"{output_path}/eligable_eids_220414.feather")
eids_dict = eligable_eids.set_index("endpoint")["eid_list"].to_dict()

In [9]:
%env MKL_NUM_THREADS=4
%env NUMEXPR_NUM_THREADS=4
%env OMP_NUM_THREADS=4

env: MKL_NUM_THREADS=4
env: NUMEXPR_NUM_THREADS=4
env: OMP_NUM_THREADS=4


In [10]:
ray.shutdown()

In [10]:
import ray

ray.init(num_cpus=24)#, dashboard_port=24762, dashboard_host="0.0.0.0", include_dashboard=True)#, webui_url="0.0.0.0"))

2022-04-21 11:55:23,126	INFO services.py:1412 -- View the Ray dashboard at http://127.0.0.1:8265


{'node_ip_address': '10.32.105.13',
 'raylet_ip_address': '10.32.105.13',
 'redis_address': None,
 'object_store_address': '/tmp/ray/session_2022-04-21_11-55-16_747114_244618/sockets/plasma_store',
 'raylet_socket_name': '/tmp/ray/session_2022-04-21_11-55-16_747114_244618/sockets/raylet',
 'webui_url': '127.0.0.1:8265',
 'session_dir': '/tmp/ray/session_2022-04-21_11-55-16_747114_244618',
 'metrics_export_port': 53571,
 'gcs_address': '10.32.105.13:46243',
 'address': '10.32.105.13:46243',
 'node_id': 'cabc44f6eee7224a80d960988028c041bcf3ab8bf9beb31213ee216f'}

In [11]:
AgeSex = ["age_at_recruitment_f21022_0_0", "sex_f31_0_0"]

# Train COX

In [12]:
in_path = pathlib.Path(f"{experiment_path}/coxph/input")
in_path.mkdir(parents=True, exist_ok=True)

model_path = f"{experiment_path}/coxph/models"
pathlib.Path(model_path).mkdir(parents=True, exist_ok=True)

In [13]:
models = ['Identity(Records)+MLP']

In [14]:
from formulaic.errors import FactorEvaluationError

In [15]:
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
import zstandard
import pickle

def get_features(endpoint):
    features = {
        'Identity(Records)+MLP': {
            "Age+Sex": AgeSex,
            "MedicalHistory": [endpoint],
            "Age+Sex+MedicalHistory": AgeSex + [endpoint],
            "Age+Sex+MedicalHistory+I(Age*MH)": AgeSex + [endpoint]
            }
    }
    return features

def get_train_data(in_path, partition, models, mapping):
    train_data = {
        model: pd.read_feather(f"{in_path}/{model}/{partition}/train.feather")\
        .set_index("eid").merge(data_outcomes, left_index=True, right_index=True, how="left").replace(mapping)
        for model in models}
    return train_data

def fit_cox(data_fit, feature_set, covariates, endpoint, penalizer, step_size=1):
    if feature_set=="Age+Sex+MedicalHistory+I(Age*MH)":
        endpoint_label = endpoint.replace("-", "")
        data_fit.columns = [c.replace("-", "") for c in data_fit.columns]
        covariates = [c.replace("-", "") for c in covariates]
        #print(endpoint_label)
        #print(data_fit)
        #print(covariates)
        if "sex_f31_0_0" in covariates:
            formula=f"age_at_recruitment_f21022_0_0*{endpoint_label}+sex_f31_0_0*{endpoint_label}"
        else:
            formula=f"age_at_recruitment_f21022_0_0*{endpoint_label}"
        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(data_fit, f"{endpoint_label}_time", f"{endpoint_label}_event", formula=formula, step_size=step_size)
    else:
        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(data_fit, f"{endpoint}_time", f"{endpoint}_event", step_size=step_size)

    return cph

def save_pickle(data, data_path):
    with open(data_path, "wb") as fh:
        cctx = zstandard.ZstdCompressor()
        with cctx.stream_writer(fh) as compressor:
            compressor.write(pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL))
            
def load_pickle(fp):
    with open(fp, "rb") as fh:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(fh) as decompressor:
            data = pickle.loads(decompressor.read())
    return data

@ray.remote
def fit_endpoint(data_partition, eids_dict, endpoint_defs, endpoint, partition, models, model_path):
    eids_incl = eids_dict[endpoint].tolist()
    features = get_features(endpoint)
    eligibility = endpoint_defs.loc[endpoint]["sex"]
    for model in models:
        data_model = data_partition[model]
        for feature_set, covariates in features[model].items():
            cph_path = f"{model_path}/{endpoint}_{feature_set}_{partition}.p"
            if os.path.isfile(cph_path):
                try:
                    cph = load_pickle(cph_path)
                    success = True
                except:
                    success = False
                    pass
            if not os.path.isfile(cph_path) or success==False:
                if (eligibility != "Both") and ("sex_f31_0_0" in covariates): 
                    covariates = [c for c in covariates if c!="sex_f31_0_0"]
                data_endpoint = data_model[covariates + [f"{endpoint}_event", f"{endpoint}_time"]].astype(np.float32)
                data_endpoint = data_endpoint[data_endpoint.index.isin(eids_incl)]
                try:
                    cph = fit_cox(data_endpoint, feature_set, covariates, endpoint, penalizer=0.0)
                    save_pickle(cph, cph_path)
                except (ValueError, ConvergenceError, KeyError,FactorEvaluationError) as e:
                    print("ConvergenceError", model, endpoint, feature_set, partition, "problem: reduce step size")
                    try:
                        cph = fit_cox(data_endpoint, feature_set, covariates, endpoint, penalizer=0.0, step_size=0.5)
                        save_pickle(cph, cph_path)
                        print("ConvergenceError", model, endpoint, feature_set, partition, "trying with reduced step size ... 0.5 successfull")
                    except (ValueError, ConvergenceError, KeyError,FactorEvaluationError) as e:
                        print("ConvergenceError", model, endpoint, feature_set, partition, "trying with reduced step size ... 0.5 failed")
                        try:
                            cph = fit_cox(data_endpoint, feature_set, covariates, endpoint, penalizer=0.0, step_size=0.1)
                            save_pickle(cph, cph_path)
                            print("ConvergenceError", model, endpoint, feature_set, partition, "trying with reduced step size ... 0.1 successfull")
                        except (ValueError, ConvergenceError, KeyError, FactorEvaluationError) as e:
                            print("ConvergenceError", model, endpoint, feature_set, partition, "trying with reduced step size ... 0.1 failed")
                            save_pickle(data_endpoint, f"{experiment_path}/coxph/errordata_{endpoint}_{feature_set}_{partition}.p")
                            pass
    return True

In [16]:
model_list =  !ls $model_path
model_list = [m for m in model_list if "I(" in m]

In [18]:
model_list

['OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_0.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_1.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_10.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_11.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_12.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_13.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_14.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_15.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_16.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_17.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_18.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_19.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_2.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_20.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_21.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_3.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_4.p',
 'OMOP_4306655_Age+Sex+MedicalHistory+I(Age*MH)_5.p',
 'OMOP_4306655_A

In [ ]:
1+1

In [ ]:
mapping = {"sex_f31_0_0": {"Female":0, "Male":1}}

ray_eids = ray.put(eids_dict)
ray_endpoint_defs = ray.put(endpoint_defs)
for partition in tqdm(partitions):
    try:
        del ray_partition
    except:
        print("Ray object not yet initialised")
    try:
        data_partition = get_train_data(in_path, partition, models, mapping)
        ray_partition = ray.put(data_partition)
        progress = []
        for endpoint in endpoints:
            progress.append(fit_endpoint.remote(ray_partition, ray_eids, ray_endpoint_defs, endpoint, partition, models, model_path))
        [ray.get(s) for s in tqdm(progress)]
    except FileNotFoundError:
        pass

In [32]:
load_pickle("/sc-projects/sc-proj-ukb-cvd/results/projects/22_medical_records/data/220413/coxph/errordata_phecode_002-1_Age+Sex+MedicalHistory+I(Age*MH)_0.p")

,age_at_recruitment_f21022_0_0,sex_f31_0_0,phecode_002-1,phecode_002-1_event,phecode_002-1_time
eid,,,,,
1000018,-0.931072,0.0,0.086565,0.0,11.866089
1000020,0.304493,1.0,-1.797872,0.0,13.596446
1000037,0.304493,0.0,0.259373,0.0,12.868163
1000043,0.798719,1.0,2.246452,0.0,12.309629
1000051,-0.683959,0.0,0.685702,0.0,15.291210
...,...,...,...,...,...
5337676,0.922275,1.0,0.725549,0.0,11.518374
5337681,1.292945,1.0,0.346841,0.0,12.312367
5337697,-1.919524,1.0,0.265374,0.0,11.822283


In [22]:
data_partition['Identity(Records)+MLP']['phecode_977']

eid
1303905    0.507814
1303918    0.033095
1303920   -0.946619
1303937    0.042184
1303943   -0.270619
             ...   
5316968   -0.614777
5316970    0.009382
5316985   -0.361461
5316994   -0.194883
5317002   -0.581893
Name: phecode_977, Length: 401263, dtype: float64